# Project: Customer Support System

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) CrewAI roadmap.

You will build a triage → conditional specialist routing → QA pipeline using Flows, ConditionalTask, Knowledge, a custom tool, structured outputs, human input, and memory.

## Install dependencies

In [ ]:
!pip install -q crewai crewai-tools

## API key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Step 1: Data models

`SupportTicket` is produced by triage; `SupportResponse` is the QA deliverable.

In [ ]:
from pydantic import BaseModel


class SupportTicket(BaseModel):
    category: str  # "billing", "technical", "general"
    priority: str  # "low", "medium", "high"
    summary: str


class SupportResponse(BaseModel):
    ticket_summary: str
    resolution: str
    follow_up_needed: bool
    satisfaction_prediction: str

## Step 2: FAQ knowledge source

`StringKnowledgeSource` feeds crew-level RAG for policy-style answers.

In [ ]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

faq = StringKnowledgeSource(
    content=(
        "Q: How do I reset my password? A: Go to Settings > Security > Reset password and follow the email link. "
        "Q: I was charged twice. A: Open Billing > Invoices, note the duplicate charge ids, and request a refund review. "
        "Q: The app returns error 503. A: Check status.example.com; if green, try clearing cache or reinstalling the app."
    ),
)

## Step 3: Custom tool — ticket lookup

Specialists call this when the customer cites a ticket id.

In [ ]:
from crewai.tools import tool


@tool("Ticket Lookup")
def lookup_ticket(ticket_id: str) -> str:
    """Look up a support ticket by id in the CRM (simulated)."""
    crm = {
        "T-1001": "Status: open. Issue: duplicate subscription charge from 2026-03-10.",
        "T-2044": "Status: waiting on customer. Issue: SMTP errors when sending reports.",
    }
    return crm.get(ticket_id.strip().upper(), "No ticket found for that id.")

## Step 4: Agents

Triage, billing and technical specialists (with `lookup_ticket`), and a QA reviewer.

In [ ]:
from crewai import Agent

triage_agent = Agent(
    role="Support Triage",
    goal="Classify incoming messages into category, priority, and a short internal summary",
    backstory="You scan customer messages quickly and assign accurate routing labels without solving the full case.",
    verbose=True,
    allow_delegation=False,
)

billing_specialist = Agent(
    role="Billing Specialist",
    goal="Resolve billing, invoices, refunds, and plan questions using policy and tools",
    backstory="You know subscription billing inside out and escalate edge cases clearly.",
    tools=[lookup_ticket],
    verbose=True,
    allow_delegation=False,
)

technical_specialist = Agent(
    role="Technical Specialist",
    goal="Diagnose product and integration issues with concrete steps and expectations",
    backstory="You reproduce issues mentally, cite known error patterns, and avoid guessing when data is missing.",
    tools=[lookup_ticket],
    verbose=True,
    allow_delegation=False,
)

qa_reviewer = Agent(
    role="Quality Assurance Reviewer",
    goal="Ensure the proposed resolution is accurate, on-brand, and complete before it reaches the customer",
    backstory="You edit for clarity, policy alignment, and risk; you catch contradictions with the FAQ.",
    verbose=True,
    allow_delegation=False,
)

## Step 5: Tasks — triage, ConditionalTask routes, QA

Conditions read `category` from the **previous executed** task output (still triage when earlier branches skip). QA uses `context` from the three specialist tasks and `human_input=True`.

In [ ]:
from crewai import Crew, Process, Task
from crewai.tasks.conditional_task import ConditionalTask
from crewai.tasks.task_output import TaskOutput


def _category_from_previous(output: TaskOutput) -> str | None:
    text = output.raw.lower()
    for cat in ("billing", "technical", "general"):
        if (
            f'"category": "{cat}"' in text
            or f"'category': '{cat}'" in text
            or f'category":"{cat}"' in text
        ):
            return cat
    return None


def is_billing(output: TaskOutput) -> bool:
    return _category_from_previous(output) == "billing"


def is_technical(output: TaskOutput) -> bool:
    return _category_from_previous(output) == "technical"


def is_general(output: TaskOutput) -> bool:
    return _category_from_previous(output) == "general"


triage_task = Task(
    description=(
        "Customer message: {customer_query}\n"
        "Return a SupportTicket: category must be exactly one of billing, technical, general; "
        "priority one of low, medium, high; summary in one short paragraph."
    ),
    expected_output="A validated SupportTicket",
    agent=triage_agent,
    output_pydantic=SupportTicket,
)

billing_task = ConditionalTask(
    description=(
        "Resolve this billing case. Message: {customer_query}. "
        "Use FAQ knowledge. If a ticket id like T-1001 appears, call lookup_ticket. "
        "Write the resolution you would send to the customer (plain text)."
    ),
    expected_output="Customer-ready billing resolution",
    agent=billing_specialist,
    condition=is_billing,
)

technical_task = ConditionalTask(
    description=(
        "Resolve this technical case. Message: {customer_query}. "
        "Use FAQ knowledge and lookup_ticket if an id is present. "
        "Write numbered troubleshooting steps and what to do if they fail."
    ),
    expected_output="Customer-ready technical resolution",
    agent=technical_specialist,
    condition=is_technical,
)

general_task = ConditionalTask(
    description=(
        "Handle this general inquiry. Message: {customer_query}. "
        "Use FAQ knowledge. Answer helpfully and state any limits clearly."
    ),
    expected_output="Customer-ready general answer",
    agent=triage_agent,
    condition=is_general,
)

qa_task = Task(
    description=(
        "Review the specialist output (from billing, technical, or general handling). "
        "Produce SupportResponse: ticket_summary (brief), resolution (final customer-facing text), "
        "follow_up_needed (bool), satisfaction_prediction (low|medium|high)."
    ),
    expected_output="A validated SupportResponse",
    agent=qa_reviewer,
    context=[billing_task, technical_task, general_task],
    output_pydantic=SupportResponse,
    human_input=True,
)

## Step 6: Crew + Flow

The crew enables `knowledge_sources`, `memory=True`, and sequential execution. The `Flow` wraps `kickoff` so you can extend orchestration later.

In [ ]:
from crewai.flow.flow import Flow, start

support_crew = Crew(
    agents=[triage_agent, billing_specialist, technical_specialist, qa_reviewer],
    tasks=[triage_task, billing_task, technical_task, general_task, qa_task],
    process=Process.sequential,
    knowledge_sources=[faq],
    memory=True,
    verbose=True,
)


class SupportFlowState(BaseModel):
    customer_query: str = ""


class CustomerSupportFlow(Flow[SupportFlowState]):
    @start()
    def run_support_crew(self):
        return support_crew.kickoff(inputs={"customer_query": self.state.customer_query})

## Step 7: Run a sample query

You will be prompted for human input when QA runs. Swap `customer_query` to exercise technical or general routing.

In [ ]:
flow = CustomerSupportFlow()
final = flow.kickoff(
    inputs={"customer_query": "I was charged twice—my ticket is T-1001. Please fix it."}
)
resp = final.pydantic
print(resp.ticket_summary)
print(resp.resolution)
print(resp.follow_up_needed, resp.satisfaction_prediction)

## What you learned

- **Structured triage** with `output_pydantic=SupportTicket` and specialist plus QA schemas.
- **ConditionalTask** branching driven by triage `category`, compatible with sequential “last executed output” behavior.
- **Knowledge** via `StringKnowledgeSource` at the crew level.
- A **custom `@tool`** for CRM-style ticket lookup.
- **Memory** on the crew and **human input** on QA before the final `SupportResponse`.
- A **Flow** as the public entrypoint around `support_crew.kickoff(...)`.